In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import RobustScaler, QuantileTransformer
from ctgan import CTGAN
from scipy.stats import ks_2samp
from scipy.spatial.distance import jensenshannon
import warnings
warnings.filterwarnings('ignore')
import os

# 1. 数据加载和预处理
df = pd.read_excel('../data/quyi_data_x_names.xlsx')
if len(df) != 1400:
    raise ValueError('Expected 1,400 ordered furnace records (900 train + 500 test).')
train_df = df.iloc[:900].copy()
holdout_df = df.iloc[900:].copy()
print(f"原始数据集大小: {df.shape}")

# 分离特征和多个目标变量
X = train_df.drop(['x25', 'x26', 'x27', 'x28', 'x29'], axis=1)
y = train_df[['x25', 'x26', 'x27', 'x28', 'x29']]

# 将目标变量合并回特征用于GAN训练
data_for_gan = X.copy()
for col in y.columns:
    data_for_gan[col] = y[col]

# 2. 分组特征，然后单独训练每组特征
# 相关性分析以确定特征分组
corr_matrix = data_for_gan.corr().abs()

# 特征分组方法：根据相关性分成3组
def create_feature_groups(corr_matrix, n_groups=3, threshold=0.4):
    remaining_features = set(corr_matrix.columns)
    groups = []
    
    # 尝试基于相关性创建组
    while remaining_features and len(groups) < n_groups:
        group = []
        # 随机选择一个起始特征
        if not group:
            start_feature = np.random.choice(sorted(remaining_features))
            group.append(start_feature)
            remaining_features.remove(start_feature)
        
        # 添加与组内特征高度相关的其他特征
        features_to_add = []
        for feature in remaining_features:
            # 检查与当前组中任何特征的相关性
            for f in group:
                if corr_matrix.loc[f, feature] >= threshold:
                    features_to_add.append(feature)
                    break
        
        for feature in features_to_add:
            if feature in remaining_features:  # 确保特征仍在剩余特征集中
                group.append(feature)
                remaining_features.remove(feature)
        
        if group:  # 只有当组不为空时才添加
            groups.append(group)
    
    # 将剩余特征平均分配到已有的组中
    if remaining_features:
        remaining_list = sorted(remaining_features)
        for i, feature in enumerate(remaining_list):
            groups[i % len(groups)].append(feature)
    
    return groups

# 创建特征组
np.random.seed(42)  # 设置随机种子以确保可重复性
feature_groups = create_feature_groups(corr_matrix)
print(f"特征已分成 {len(feature_groups)} 组")
for i, group in enumerate(feature_groups):
    print(f"组 {i+1}: {group}")

# 3. 为每个组训练单独的CTGAN模型
def transform_data(data):
    """使用分位数变换进行更高级的预处理"""
    transformer = QuantileTransformer(output_distribution='normal', random_state=42)
    data_transformed = transformer.fit_transform(data)
    return pd.DataFrame(data_transformed, columns=data.columns), transformer

def inverse_transform_data(data_transformed, transformer):
    """反向转换数据"""
    return pd.DataFrame(transformer.inverse_transform(data_transformed), 
                        columns=data_transformed.columns)

# 存储每个组的生成数据
synthetic_data_parts = []

for i, group in enumerate(feature_groups):
    print(f"\n处理特征组 {i+1}/{len(feature_groups)}")
    
    # 获取此组的数据
    group_data = data_for_gan[group].copy()
    
    # 转换数据
    group_data_transformed, transformer = transform_data(group_data)
    
    # 配置并训练CTGAN
    ctgan = CTGAN(
        embedding_dim=128,
        generator_dim=(256, 256, 256),  # 更深的网络
        discriminator_dim=(256, 256, 256),  # 更深的网络
        generator_lr=1e-4, 
        discriminator_lr=1e-4,
        batch_size=100,  # 适当调整批量大小
        epochs=1500,      # 显著增加训练轮次
        pac=10,
        verbose=True
    )
    
    print(f"开始为组 {i+1} 训练CTGAN模型...")
    ctgan.fit(group_data_transformed)
    print(f"组 {i+1} 的CTGAN模型训练完成！")
    
    # 生成合成数据
    num_samples = 2*len(group_data)+300
    group_synthetic_transformed = ctgan.sample(num_samples)
    
    # 反向转换
    group_synthetic = inverse_transform_data(group_synthetic_transformed, transformer)
    
    # 后处理：修正异常值
    for col in group:
        col_min = data_for_gan[col].min()
        col_max = data_for_gan[col].max()
        
        # 扩展范围以允许一些新颖性
        extend_ratio = 0.05
        extended_min = col_min - (col_max - col_min) * extend_ratio
        extended_max = col_max + (col_max - col_min) * extend_ratio
        
        # 裁剪超出范围的值
        group_synthetic[col] = np.clip(group_synthetic[col], extended_min, extended_max)
    
    # 保存这个组的生成数据
    synthetic_data_parts.append(group_synthetic)

# 4. 合并所有组的生成数据
# 第一步：确保所有组的数据具有相同的行数
min_rows = min(len(df) for df in synthetic_data_parts)
synthetic_data_parts_aligned = [df.iloc[:min_rows] for df in synthetic_data_parts]

# 第二步：合并所有组
synthetic_data = pd.concat(synthetic_data_parts_aligned, axis=1)

# 确保列的顺序与原始数据相同
synthetic_data = synthetic_data[data_for_gan.columns]

print(f"\n生成的合成数据大小: {synthetic_data.shape}")

# 5. 计算相关矩阵和评估指标
corr_original = data_for_gan.corr()
corr_synthetic = synthetic_data.corr()
corr_diff = np.abs(corr_original - corr_synthetic)
print(f"相关矩阵平均绝对差异: {np.mean(corr_diff):.4f}")

# 计算KS检验和JS散度
def compute_js_divergence(p, q, bins=50):  # 增加bins以获得更精确的分布
    # 创建直方图以获取概率分布
    hist_p, bin_edges = np.histogram(p, bins=bins, density=True)
    hist_q, _ = np.histogram(q, bins=bin_edges, density=True)
    
    # 添加小的平滑值防止零概率
    hist_p = hist_p + 1e-10
    hist_q = hist_q + 1e-10
    
    # 归一化
    hist_p = hist_p / np.sum(hist_p)
    hist_q = hist_q / np.sum(hist_q)
    
    # 计算JS散度
    return jensenshannon(hist_p, hist_q)

def distribution_similarity(real_data, synthetic_data, columns):
    results = {}
    avg_ks = 0
    avg_js = 0
    
    print("\n分布相似性评估:")
    print("-" * 50)
    print(f"{'特征':>8} | {'KS统计量':>10} | {'p值':>10} | {'JS散度':>10}")
    print("-" * 50)
    
    for col in columns:
        # KS检验
        ks_statistic, p_value = ks_2samp(real_data[col], synthetic_data[col])
        
        # JS散度
        js_div = compute_js_divergence(real_data[col], synthetic_data[col])
        
        results[col] = {
            'KS_statistic': ks_statistic,
            'p_value': p_value,
            'JS_divergence': js_div
        }
        
        avg_ks += ks_statistic
        avg_js += js_div
        
        print(f"{col:>8} | {ks_statistic:>10.4f} | {p_value:>10.4f} | {js_div:>10.4f}")
    
    avg_ks /= len(columns)
    avg_js /= len(columns)
    
    print("-" * 50)
    print(f"{'平均':>8} | {avg_ks:>10.4f} | {'':>10} | {avg_js:>10.4f}")
    print("-" * 50)
    print("KS统计量越小越好，接近0表示分布相似")
    print("JS散度越小越好，接近0表示分布相似")
    
    return results, avg_ks, avg_js

# 计算所有变量的分布相似性
similarity_metrics, avg_ks, avg_js = distribution_similarity(data_for_gan, synthetic_data, data_for_gan.columns)

# 6. 合并原始数据和生成的数据
combined_data = pd.concat([data_for_gan, synthetic_data], axis=0)
print(f"合并后数据集大小: {combined_data.shape}")

# 7. 保存生成的数据
combined_data.to_excel('../data/CTGAN_grouped_data_multi.xlsx', index=False)
holdout_df.to_excel('../data/holdout_data.xlsx', index=False)

# 8. 生成评估结论
print("\n==== 生成数据质量评估结论 ====")
print(f"1. 平均KS统计量: {avg_ks:.4f} (越接近0越好)")
print(f"2. 平均JS散度: {avg_js:.4f} (越接近0越好)")
print(f"3. 相关矩阵平均绝对差异: {np.mean(corr_diff):.4f} (越接近0越好)")
print("4. 使用特征分组和高级预处理提高了生成数据质量")
print("5. 每组特征使用专门的CTGAN模型训练")
print("6. 生成数据集大小提升: {:.1f}%".format((len(combined_data)/len(data_for_gan) - 1)*100))

# 基于经验的评分标准
ks_score = 5 - min(5, avg_ks * 10)  # KS < 0.1 为最佳
js_score = 5 - min(5, avg_js * 10)  # JS < 0.1 为最佳
corr_score = 5 - min(5, np.mean(corr_diff) * 10)  # 差异 < 0.1 为最佳

total_score = (ks_score + js_score + corr_score) / 3
print("\n综合质量评分: {:.1f}/5.0".format(total_score))

if total_score >= 4.5:
    print("评价: 优秀 - 生成数据非常接近原始分布")
elif total_score >= 4.0:
    print("评价: 良好 - 生成数据接近原始分布")
elif total_score >= 3.0:
    print("评价: 中等 - 生成数据保留了原始数据的主要特征")
elif total_score >= 2.0:
    print("评价: 一般 - 生成数据与原始数据有明显差异")
else:
    print("评价: 较差 - 建议调整GAN参数或训练轮次")